In [1]:
# =========================================
# IMPORTS
# =========================================
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree


# =========================================
# USER INPUT
# =========================================
# RUN_FOLDER = "Hexa_no_Advection_single_Stokes"
# RUN_FOLDER = "Hexa_no_Advection_iterated_Stokes"
# RUN_FOLDER ="only_Landlab_uplift_diffussion_no_Advection_no_Stokes"
# RUN_FOLDER ="only_Landlab_x-900_y-1900_diff_5e3_n_subtsteps-20_uplift_diffussion_no_Advection_no_Stokes"


# RUN_FOLDER ="xy-repition_1_only_Landlab_x-900_y-1900_no_Advection_no_Stokes"

# RUN_FOLDER ="Linear_slope_substep_10_diffu-1_xy-repition_0_global_refin-0_only_Landlab_x-900_y-1900_z-1000_no_Advection_no_Stokes"
# RUN_FOLDER = "Linear_slope_substep_10_diffu-1_xy-repition_0_global_refin-0_only_Landlab_x-900_y-1900_z-1000_no_Advection_no_Stokes"
# RUN_FOLDER ="Linear_Time_test_D_second_s2yr_no_Advection_no_Stokes"
# RUN_FOLDER ="global_refin-1_Diffusion_Zero_substep_10_xy-repition_5_x-900_y-1900_z-1000_no_Advection_no_Stokes"
# RUN_FOLDER ="global_refin-3_Diffusion_5e4_m2-YEAR_substep_10_xy-repition_5_x-900_y-1900_z-1000_no_Advection_no_Stokes"

# RUN_FOLDER ="global_refin-0_x_27_y_57-repition_Diffusion_0_substep_10_x-900_y-1900_z-1000_no_Advection_no_Stokes"
# RUN_FOLDER = "global_refin-0_x_2_y_2-repition_Diffusion_5e4_m2_per_Year_substep_10_x-900_y-1900_z-1000_no_Advection_no_Stokes"
RUN_FOLDER = "Landlab_defined_uplift_subset_10_global_refin-0_x_9_y_19-repition_Diffusion_5e4_m2_per_Year_Set_Year-false_x-900_y-1900_z-1000_no_Advection_no_Stokes"
# =========================================
# PATH SETUP
# =========================================
BASE_DIR = os.getcwd()

RUN_DIR = os.path.join(
    BASE_DIR,
    "outputs",
    RUN_FOLDER
)

TOPOGRAPHY_DIR = os.path.join(RUN_DIR, "topography")

LANDLAB_DIR = os.path.join(RUN_DIR, "landlab")

RESULT_DIR = os.path.join(RUN_DIR, "results_elevation_R2")

os.makedirs(RESULT_DIR, exist_ok=True)


# =========================================
# PARAMETERS
# =========================================
BASE_TOL = 1e-6
MIN_OVERLAP_PERCENT = 20.0
RMSE_THRESHOLD = 1e-3
MAX_DIFF_THRESHOLD = 1e-2


# =========================================
# HELPERS
# =========================================
def get_step(fname):
    nums = re.findall(r"\d+", fname)
    return int(nums[-1]) if nums else -1


# =========================================
# LOAD LANDLAB VTK
# =========================================
def load_landlab(file_path):

    with open(file_path, "r") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):

        if line.startswith("POINTS"):

            n = int(line.split()[1])
            start = i + 1
            break

    data = np.array(
        [list(map(float, l.split())) for l in lines[start:start+n]]
    )

    return data[:, 0], data[:, 1], data[:, 2]


# =========================================
# LOAD ASPECT TOPOGRAPHY
# =========================================
def load_aspect(file_path):

    data = np.loadtxt(file_path, comments="#")

    return data[:, 0], data[:, 1], data[:, 3]


# =========================================
# VALIDATION FUNCTION
# =========================================
def validate_pair(x1, y1, z1, x2, y2, z2):

    pts1 = np.column_stack((x1, y1))
    pts2 = np.column_stack((x2, y2))

    tree = cKDTree(pts2)

    dist, idx = tree.query(pts1)

    tol_adapt = (
        max(BASE_TOL, np.percentile(dist, 10))
        if len(dist) > 0 else BASE_TOL
    )

    overlap_mask = dist < tol_adapt

    # -------------------------------------
    # OVERLAP %
    # -------------------------------------
    N_total = len(x1)

    overlap_percent = (
        np.sum(overlap_mask) / N_total * 100
        if N_total > 0 else 0.0
    )

    # -------------------------------------
    # DOMAIN CHECK
    # -------------------------------------
    xmin, xmax = np.min(x2), np.max(x2)
    ymin, ymax = np.min(y2), np.max(y2)

    outside_mask = (
        (x1 < xmin) |
        (x1 > xmax) |
        (y1 < ymin) |
        (y1 > ymax)
    )

    inside_mask = ~outside_mask

    outside_percent = (
        np.sum(outside_mask) / N_total * 100
        if N_total > 0 else 0.0
    )

    # -------------------------------------
    # OVERLAP STATISTICS
    # -------------------------------------
    if np.any(overlap_mask):

        z1_o = z1[overlap_mask]
        z2_o = z2[idx][overlap_mask]

        diff = z1_o - z2_o

        rmse_o = np.sqrt(np.mean(diff**2))

        max_o = np.max(np.abs(diff))

        r2_o = (
            1
            - np.sum(diff**2)
            / np.sum((z1_o - np.mean(z1_o))**2)
        )

    else:

        rmse_o = np.nan
        max_o = np.nan
        r2_o = np.nan

    # -------------------------------------
    # DOMAIN STATISTICS
    # -------------------------------------
    if np.any(inside_mask):

        z1_d = z1[inside_mask]
        z2_d = z2[idx][inside_mask]

        diff = z1_d - z2_d

        rmse_d = np.sqrt(np.mean(diff**2))

        max_d = np.max(np.abs(diff))

        r2_d = (
            1
            - np.sum(diff**2)
            / np.sum((z1_d - np.mean(z1_d))**2)
        )

    else:

        rmse_d = np.nan
        max_d = np.nan
        r2_d = np.nan

    # -------------------------------------
    # PASS / FAIL
    # -------------------------------------
    passed = True

    if overlap_percent < MIN_OVERLAP_PERCENT:
        passed = False

    if rmse_o > RMSE_THRESHOLD:
        passed = False

    if max_o > MAX_DIFF_THRESHOLD:
        passed = False

    return {

        "overlap_percent": overlap_percent,
        "outside_percent": outside_percent,

        "rmse_overlap": rmse_o,
        "max_diff_overlap": max_o,
        "r2_overlap": r2_o,

        "rmse_domain": rmse_d,
        "max_diff_domain": max_d,
        "r2_domain": r2_d,

        "overlap_mask": overlap_mask,
        "inside_mask": inside_mask,
        "outside_mask": outside_mask,

        "idx": idx,

        "passed": passed
    }


# =========================================
# FILE COLLECTION
# =========================================
landlab_files = sorted(

    [
        f for f in os.listdir(LANDLAB_DIR)
        if f.endswith(".vtk")
    ],

    key=get_step
)

aspect_files = sorted(

    [
        f for f in os.listdir(TOPOGRAPHY_DIR)
        if os.path.isfile(os.path.join(TOPOGRAPHY_DIR, f))
    ],

    key=get_step
)


# =========================================
# DEBUG PRINT
# =========================================
print("\nLANDLAB FILES")
print("-------------------")

for f in landlab_files:
    print(f)

print("\nASPECT FILES")
print("-------------------")

for f in aspect_files:
    print(f)


# =========================================
# PAIRS
# =========================================
pairs = []

# ---- Zero timestep ----
if len(landlab_files) > 0 and len(aspect_files) > 0:

    pairs.append((
        "Zero timestep",
        landlab_files[0],
        aspect_files[0]
    ))

# ---- First timestep ----
if len(landlab_files) > 1 and len(aspect_files) > 1:

    pairs.append((
        "1st time step",
        landlab_files[1],
        aspect_files[1]
    ))

# ---- Middle timestep ----
if len(landlab_files) > 2 and len(aspect_files) > 2:

    pairs.append((
        "Middle time step",
        landlab_files[len(landlab_files)//2],
        aspect_files[len(aspect_files)//2]
    ))

# ---- End timestep ----
if len(landlab_files) > 0 and len(aspect_files) > 0:

    pairs.append((
        "End time step",
        landlab_files[-1],
        aspect_files[-1]
    ))


# =========================================
# PLOTTING
# =========================================
for name, lf, af in pairs:

    print("\n========================================")
    print(f"PROCESSING: {name}")
    print("========================================")

    # -------------------------------------
    # LOAD LANDLAB
    # -------------------------------------
    x1, y1, z1 = load_landlab(
        os.path.join(LANDLAB_DIR, lf)
    )

    # -------------------------------------
    # LOAD ASPECT
    # -------------------------------------
    x2, y2, z2 = load_aspect(
        os.path.join(TOPOGRAPHY_DIR, af)
    )

    # -------------------------------------
    # REMOVE DUPLICATE ASPECT NODES
    # -------------------------------------
    total_rows = len(x2)

    xy = np.column_stack((x2, y2))

    xy_unique, unique_idx = np.unique(
        xy,
        axis=0,
        return_index=True
    )

    unique_nodes = len(xy_unique)

    x2 = x2[unique_idx]
    y2 = y2[unique_idx]
    z2 = z2[unique_idx]

    nx = len(np.unique(x2))
    ny = len(np.unique(y2))

    # -------------------------------------
    # PRINT INFO
    # -------------------------------------
    print(f"ASPECT FILE         : {af}")
    print(f"Raw rows            : {total_rows}")
    print(f"Unique nodes        : {unique_nodes}")
    print(f"Grid size           : {nx} x {ny}")

    # -------------------------------------
    # VALIDATION
    # -------------------------------------
    result = validate_pair(
        x1, y1, z1,
        x2, y2, z2
    )

    # -------------------------------------
    # NODE COUNTS
    # -------------------------------------
    n_landlab = len(x1)
    n_aspect = len(x2)
    n_total = n_landlab + n_aspect

    # -------------------------------------
    # DOMAIN EXTENTS
    # -------------------------------------
    ll_x_extent = np.max(x1) - np.min(x1)
    ll_y_extent = np.max(y1) - np.min(y1)

    asp_x_extent = np.max(x2) - np.min(x2)
    asp_y_extent = np.max(y2) - np.min(y2)

    # =====================================
    # FIGURE
    # =====================================
    plt.figure(figsize=(12, 10))

    # =====================================
    # OVERLAP
    # =====================================
    plt.subplot(2, 2, 1)

    z1_o = z1[result["overlap_mask"]]
    z2_o = z2[result["idx"]][result["overlap_mask"]]

    if len(z1_o) > 0:

        plt.scatter(
            z2_o,
            z1_o,
            s=10,
            label=f"Points (N={len(z1_o)})"
        )

        m1 = min(z2_o.min(), z1_o.min())
        m2 = max(z2_o.max(), z1_o.max())

        plt.plot(
            [m1, m2],
            [m1, m2],
            "r--",
            label="1:1 line"
        )

    plt.xlabel("ASPECT Elevation")
    plt.ylabel("Landlab Elevation")

    plt.title(
        "Overlap\n\n"
        "Elevation comparison at spatially matching\n"
        "nodes between Landlab and ASPECT"
    )

    plt.legend(
        title=f"R² = {result['r2_overlap']:.4f}"
    )

    # =====================================
    # DOMAIN
    # =====================================
    plt.subplot(2, 2, 2)

    z1_d = z1[result["inside_mask"]]
    z2_d = z2[result["idx"]][result["inside_mask"]]

    if len(z1_d) > 0:

        plt.scatter(
            z2_d,
            z1_d,
            s=10,
            label=f"Points (N={len(z1_d)})"
        )

        m1 = min(z2_d.min(), z1_d.min())
        m2 = max(z2_d.max(), z1_d.max())

        plt.plot(
            [m1, m2],
            [m1, m2],
            "r--",
            label="1:1 line"
        )

    plt.xlabel("ASPECT Elevation")
    plt.ylabel("Landlab Elevation")

    plt.title(
        "Domain\n\n"
        "Landlab points inside ASPECT domain"
    )

    plt.legend(
        title=f"R² = {result['r2_domain']:.4f}"
    )

    # =====================================
    # OVERLAY
    # =====================================
    plt.subplot(2, 2, 3)

    stride = max(1, len(x1)//5000)

    plt.scatter(
        x1[::stride],
        y1[::stride],
        s=5,
        alpha=0.5,
        label="Landlab"
    )

    plt.scatter(
        x2[::stride],
        y2[::stride],
        s=5,
        alpha=0.5,
        label="ASPECT"
    )

    plt.scatter(
        x1[result["overlap_mask"]][::stride],
        y1[result["overlap_mask"]][::stride],
        s=10,
        color="black",
        label="Exact overlap"
    )

    plt.scatter(
        x1[result["outside_mask"]][::stride],
        y1[result["outside_mask"]][::stride],
        s=10,
        color="green",
        label="Outside"
    )

    plt.legend()

    plt.title("Grid Overlay")

    plt.gca().set_aspect("equal")

    # =====================================
    # SUMMARY PANEL
    # =====================================
    plt.subplot(2, 2, 4)

    plt.axis("off")

    left = (
        "TEST RESULT\n"
        "------------------\n\n"

        f"{lf}\n"
        f"{af}\n\n"

        f"Overlap: {result['overlap_percent']:.2f}%\n"
        f"Outside: {result['outside_percent']:.2f}%\n\n"

        f"RMSE (Overlap): "
        f"{result['rmse_overlap']:.2e}\n"

        f"Max  (Overlap): "
        f"{result['max_diff_overlap']:.2e}\n\n"

        f"RMSE (Domain): "
        f"{result['rmse_domain']:.2e}\n"

        f"Max  (Domain): "
        f"{result['max_diff_domain']:.2e}\n\n"

        f"R² (Overlap): "
        f"{result['r2_overlap']:.4f}\n"

        f"R² (Domain): "
        f"{result['r2_domain']:.4f}\n\n"

        f"FINAL: "
        f"{'PASS' if result['passed'] else 'FAIL'}\n"
    )

    right = (
        # f"{RUN_FOLDER}\n"
        # "====================\n\n"

        "THRESHOLD CRITERIA\n"
        "------------------\n"

        f"Overlap ≥ "
        f"{MIN_OVERLAP_PERCENT:.0f}%\n"

        f"RMSE ≤ "
        f"{RMSE_THRESHOLD:.0e}\n"

        f"Max ≤ "
        f"{MAX_DIFF_THRESHOLD:.0e}\n\n"

        "NODE INFORMATION\n"
        "------------------\n"

        f"ASPECT Nodes   : {n_aspect}\n"
        f"Landlab Nodes  : {n_landlab}\n"
        f"Total Nodes    : {n_total}\n\n"

        "DOMAIN EXTENT (meters)\n"
        "------------------\n"

        f"ASPECT   X: {asp_x_extent:.2f} m\n"
        f"ASPECT   Y: {asp_y_extent:.2f} m\n"

        f"Landlab  X: {ll_x_extent:.2f} m\n"
        f"Landlab  Y: {ll_y_extent:.2f} m\n"
    )

    plt.text(
        -0.02,
        0.95,
        left,
        va="top",
        family="monospace"
    )

    plt.text(
        0.55,
        0.95,
        right,
        va="top",
        family="monospace"
    )

    # plt.suptitle(
    #     f"{name}\n{lf} | {af}"
    # )

    plt.tight_layout()

    plt.suptitle(
    f"{RUN_FOLDER}\n\n{name}\n{lf} | {af}",
    fontsize=16,
    fontweight="bold"
)

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # =====================================
    # SAVE
    # =====================================
    safe_name = name.replace(" ", "_")

    save_path = os.path.join(
        RESULT_DIR,
        f"{safe_name}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {save_path}")

print("\n✔ Done")


LANDLAB FILES
-------------------
landlab_0001.vtk
landlab_0002.vtk
landlab_0003.vtk
landlab_0004.vtk

ASPECT FILES
-------------------
topography.00000
topography.00001
topography.00002
topography.00003

PROCESSING: Zero timestep
ASPECT FILE         : topography.00000
Raw rows            : 684
Unique nodes        : 200
Grid size           : 10 x 20
Saved: /Users/biraj/software/landlab_ASPECT_test_2/aspect/cookbooks/landlab/bug_landlab_folder/outputs/Landlab_defined_uplift_subset_10_global_refin-0_x_9_y_19-repition_Diffusion_5e4_m2_per_Year_Set_Year-false_x-900_y-1900_z-1000_no_Advection_no_Stokes/results_elevation_R2/Zero_timestep.png

PROCESSING: 1st time step
ASPECT FILE         : topography.00001
Raw rows            : 684
Unique nodes        : 200
Grid size           : 10 x 20
Saved: /Users/biraj/software/landlab_ASPECT_test_2/aspect/cookbooks/landlab/bug_landlab_folder/outputs/Landlab_defined_uplift_subset_10_global_refin-0_x_9_y_19-repition_Diffusion_5e4_m2_per_Year_Set_Year-fal